# Activity 10: Logistic Regression From Scratch

## Learning Objectives
By the end of this activity you will be able to:
- Implement the sigmoid function and understand why it maps any real input to (0, 1)
- Derive binary cross-entropy (log loss) and code it with numerical stability
- Write the gradient descent update rule for logistic regression
- Validate your implementation by comparing predictions against sklearn's LogisticRegression

---
> **Why build it from scratch?** sklearn gives you the answer; building it yourself gives you the *intuition*.  
> Understanding the math lets you debug mysterious failures that black-box models hide from you.

## Part 1 — Imports & Synthetic Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)

# WHY synthetic data?  We control the ground truth, so we know exactly
# what a perfect model should produce.  Great for unit-testing implementations.
X, y = make_classification(
    n_samples=1000,
    n_features=2,      # 2 features → easy to visualise decision boundary
    n_redundant=0,
    n_informative=2,
    random_state=42,
    n_clusters_per_class=1
)

# Standardise — logistic regression with gradient descent converges much
# faster on z-scored inputs (all features on the same scale).
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Class balance (train): {np.bincount(y_train)}")

## Part 2 — The Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

| Input  | Output         |
|--------|----------------|
| -∞     | ≈ 0            |
| 0      | 0.5            |
| +∞     | ≈ 1            |

**Why sigmoid?** We need a function that converts an unbounded real number into a probability. Sigmoid is smooth, monotonic, and differentiable everywhere — all required for gradient descent.

In [ ]:
def sigmoid(z):
    """
    Numerically stable sigmoid.
    COMMON ERROR: Using np.exp(z) / (1 + np.exp(z)) causes overflow for large z.
    DEBUG TIP:    np.clip(z, -500, 500) before exp to prevent inf/nan.
    """
    # WHY clip?  np.exp(710) overflows to inf on 64-bit floats.
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

# Visualise sigmoid
z_range = np.linspace(-10, 10, 300)
plt.figure(figsize=(8, 4))
plt.plot(z_range, sigmoid(z_range), linewidth=2.5, color='steelblue')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='Decision threshold = 0.5')
plt.axvline(0, color='gray', linestyle=':', alpha=0.6)
plt.xlabel("z  (linear combination)")
plt.ylabel("σ(z)  — probability")
plt.title("The Sigmoid Function")
plt.legend()
plt.tight_layout()
plt.show()

# Quick sanity checks
assert abs(sigmoid(0) - 0.5) < 1e-9, "sigmoid(0) must be 0.5"
assert sigmoid(1000) > 0.999,         "sigmoid(large) must approach 1"
assert sigmoid(-1000) < 0.001,        "sigmoid(-large) must approach 0"
print("Sigmoid sanity checks passed ✓")

## Part 3 — Binary Cross-Entropy (Log Loss)

$$\mathcal{L}(\hat{y}, y) = -\frac{1}{m}\sum_{i=1}^{m}\bigl[y^{(i)}\log(\hat{y}^{(i)}) + (1-y^{(i)})\log(1-\hat{y}^{(i)})\bigr]$$

**Why log loss instead of MSE?**  
MSE with sigmoid creates a non-convex loss surface — gradient descent gets stuck in local minima.  
Log loss with sigmoid is **convex** → guaranteed to find the global minimum.

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    """
    Computes mean binary cross-entropy loss.
    COMMON ERROR: log(0) = -inf → clips keep gradients finite.
    DEBUG TIP:    If loss jumps to nan, check y_pred for 0 or 1 values.
    """
    m = len(y_true)
    # WHY epsilon clip?  Prevent log(0) which gives -inf → nan gradients
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    loss = -np.mean(
        y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
    )
    return loss

# Verify behaviour
print("Perfect predictions:", binary_cross_entropy(np.array([0,1]), np.array([0.0001, 0.9999])))
print("Terrible predictions:", binary_cross_entropy(np.array([0,1]), np.array([0.9999, 0.0001])))
print("Random predictions:  ", binary_cross_entropy(np.array([0,1]), np.array([0.5, 0.5])))

## Part 4 — Gradient Descent for Logistic Regression

The forward pass and gradient:

1. $z = X\mathbf{w} + b$
2. $\hat{y} = \sigma(z)$
3. $\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{m} X^T (\hat{y} - y)$
4. $\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{m} \sum (\hat{y} - y)$

Update rule: $\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial \mathcal{L}}{\partial \mathbf{w}}$

In [ ]:
class LogisticRegressionScratch:
    """
    Binary logistic regression trained with batch gradient descent.
    """
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X, y):
        m, n = X.shape
        # WHY initialise to zeros?  Logistic regression has a convex loss,
        # so any starting point converges. (Unlike neural nets where symmetry breaking matters.)
        self.weights = np.zeros(n)
        self.bias    = 0.0

        for i in range(self.n_iter):
            # --- Forward pass ---
            z        = X @ self.weights + self.bias   # (m,)
            y_hat    = sigmoid(z)                      # (m,)

            # --- Loss ---
            loss = binary_cross_entropy(y, y_hat)
            self.loss_history.append(loss)

            # --- Gradients ---
            error = y_hat - y                          # (m,)
            dw    = (X.T @ error) / m                  # (n,)
            db    = np.mean(error)                     # scalar

            # --- Update ---
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        # DEBUG TIP: Experiment with threshold to improve Precision/Recall trade-off.
        return (self.predict_proba(X) >= threshold).astype(int)

# Train
model = LogisticRegressionScratch(learning_rate=0.1, n_iterations=500)
model.fit(X_train, y_train)

# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(model.loss_history, color='crimson')
plt.xlabel("Iteration")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training Loss Curve — From-Scratch Logistic Regression")
plt.tight_layout()
plt.show()
print(f"Final training loss: {model.loss_history[-1]:.4f}")

## Part 5 — Evaluate & Compare with sklearn

In [ ]:
# --- Scratch model predictions ---
y_pred_scratch = model.predict(X_test)
acc_scratch = accuracy_score(y_test, y_pred_scratch)

# --- sklearn baseline ---
# WHY solver='lbfgs'?  It handles the same loss function but uses second-order
# curvature information (quasi-Newton), so it converges in far fewer steps.
sk_model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
sk_model.fit(X_train, y_train)
y_pred_sklearn = sk_model.predict(X_test)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)

print("=" * 45)
print(f"  From-scratch accuracy : {acc_scratch:.4f}")
print(f"  sklearn accuracy      : {acc_sklearn:.4f}")
print("=" * 45)

print("\n--- From-Scratch Classification Report ---")
print(classification_report(y_test, y_pred_scratch))
print("--- sklearn Classification Report ---")
print(classification_report(y_test, y_pred_sklearn))

## Part 6 — Decision Boundary Visualisation

In [ ]:
def plot_decision_boundary(ax, model_predict, X, y, title):
    """Plots the 2-D decision boundary for any predict function."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model_predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=20, linewidths=0.4)
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_decision_boundary(
    axes[0], model.predict,
    X_test, y_test,
    f"From Scratch  (acc={acc_scratch:.3f})"
)
plot_decision_boundary(
    axes[1], sk_model.predict,
    X_test, y_test,
    f"sklearn        (acc={acc_sklearn:.3f})"
)

plt.suptitle("Decision Boundaries: Scratch vs sklearn", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 7 — Effect of Learning Rate

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 1.0]
plt.figure(figsize=(10, 5))

for lr in learning_rates:
    m = LogisticRegressionScratch(learning_rate=lr, n_iterations=500)
    m.fit(X_train, y_train)
    label = f"lr={lr}  (final loss={m.loss_history[-1]:.3f})"
    plt.plot(m.loss_history, label=label)

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Learning Rate Comparison")
plt.legend()
plt.tight_layout()
plt.show()

# WHY does a very high lr diverge?
# The gradient step overshoots the minimum; the loss bounces instead of descending.
# COMMON ERROR: Picking lr=10.0 and wondering why loss is NaN after iteration 5.

## Summary

| Concept | From Scratch | sklearn equivalent |
|---|---|---|
| Sigmoid | `1/(1+exp(-z))` | built-in |
| Loss | Binary cross-entropy | `log_loss()` |
| Optimiser | Batch gradient descent | L-BFGS / Newton |
| Convergence | Slow (O(n) per step) | Fast (second-order) |

**Key takeaways:**
- The gradient of log loss is elegantly simple: `(ŷ - y) · X / m`
- Numerical stability requires clipping both the sigmoid input AND the log arguments
- Our scratch model matches sklearn accuracy within fractions of a percent

**Next:** Activity 11 — Logistic Regression in TensorFlow/Keras